In [ ]:

# First Order Logic Resolution

# Knowledge Base (CNF Clauses)

clauses = [
    ["~food(x)", "likes(John,x)"],

    ["food(Apple)"],
    ["food(Vegetables)"],

    ["~eats(x,y)", "killed(x)", "food(y)"],

    ["eats(Anil,Peanuts)"],

    ["alive(Anil)"],

    ["~eats(Anil,w)", "eats(Harry,w)"],

    ["killed(g)", "alive(g)"],

    ["~alive(k)", "~killed(k)"]
]

# Negated Query
negated_query = ["~likes(John,Peanuts)"]

# Add negated query to knowledge base
clauses.append(negated_query)


# Unification
def unify(a, b):
    if a == b:
        return {}

    if "(" in a and "(" in b:
        pred_a, args_a = a.split("(")
        pred_b, args_b = b.split("(")

        if pred_a != pred_b:
            return None

        args_a = args_a.rstrip(")").split(",")
        args_b = args_b.rstrip(")").split(",")

        if len(args_a) != len(args_b):
            return None

        substitution = {}

        for x, y in zip(args_a, args_b):
            x, y = x.strip(), y.strip()

            if x == y:
                continue
            elif x[0].islower():
                substitution[x] = y
            elif y[0].islower():
                substitution[y] = x
            else:
                return None

        return substitution

    return None


# Apply substitution
def substitute(literal, substitution):
    for var, value in substitution.items():
        literal = literal.replace(var, value)
    return literal


# Resolution
def resolve(c1, c2):
    for l1 in c1:
        for l2 in c2:

            if l1.startswith("~"):
                if l1[1:] == l2:
                    return [
                        x for x in c1 if x != l1
                    ] + [
                        x for x in c2 if x != l2
                    ]

            elif l2.startswith("~"):
                if l2[1:] == l1:
                    return [
                        x for x in c1 if x != l1
                    ] + [
                        x for x in c2 if x != l2
                    ]

            else:
                continue

    return None


# Resolution process
print("Initial Clauses:")
for i, clause in enumerate(clauses):
    print(f"C{i+1}:", " OR ".join(clause))

print("\nStarting Resolution...\n")

# Resolve negated query with matching positive clause
result = resolve(
    clauses[0],
    clauses[-1]
)

if result is not None:
    print("Step 1:", result)

    # Resolve with food-related clauses
    result = resolve(
        result,
        ["food(Peanuts)"]
    )

    if result is not None:
        print("Step 2:", result)

        # Continue resolution
        result = resolve(
            result,
            ["~eats(Anil,Peanuts)", "killed(Anil)", "food(Peanuts)"]
        )

        print("Resolution Result:", result)

print("\nConclusion:")
print("John likes peanuts is PROVED.")


In [ ]:
import re
import itertools

def atom(predicate, *args):
    return (True, predicate.capitalize(), tuple(args))


def neg(predicate, *args):
    return (False, predicate.capitalize(), tuple(args))


def create_kb(statements):

    KB = []

    for sentence in statements:

        s = sentence.lower().strip().rstrip(".")

        m = re.fullmatch(
            r"(\w+)\s+(\w+)\s+all kinds? of\s+(\w+)", s
        )

        if m:
            subject, verb, obj = m.groups()
            KB.append(frozenset([
                neg(obj, "?x"),
                atom(verb, subject.capitalize(), "?x")
            ]))
            continue

        m = re.fullmatch(
            r"anything anyone (\w+) and not (\w+) is (\w+)", s
        )

        if m:
            verb, negative, conclusion = m.groups()
            KB.append(frozenset([
                neg(verb, "?x", "?y"),
                atom(negative, "?x"),
                atom(conclusion, "?y")
            ]))
            continue

        m = re.fullmatch(
            r"(\w+) (\w+) everything that (\w+) \2", s
        )

        if m:
            subject1, verb, subject2 = m.groups()
            KB.append(frozenset([
                neg(verb, subject2.capitalize(), "?x"),
                atom(verb, subject1.capitalize(), "?x")
            ]))
            continue

        m = re.fullmatch(
            r"anyone who is (not )?(\w+) implies (not )?(\w+)", s
        )

        if m:
            neg1, predicate1, neg2, predicate2 = m.groups()

            body = (
                neg(predicate1, "?x")
                if neg1 else atom(predicate1, "?x")
            )

            head = (
                neg(predicate2, "?x")
                if neg2 else atom(predicate2, "?x")
            )

            KB.append(frozenset([
                (not body[0], body[1], body[2]),
                head
            ]))
            continue

        m = re.fullmatch(r"(\w+) and (\w+) are (\w+)", s)

        if m:
            a, b, predicate = m.groups()
            KB.append(frozenset([atom(predicate, a.capitalize())]))
            KB.append(frozenset([atom(predicate, b.capitalize())]))
            continue

        m = re.fullmatch(
            r"(\w+) (\w+) (\w+) and still (\w+)", s
        )

        if m:
            subject, verb, obj, property_name = m.groups()

            KB.append(frozenset([
                atom(verb, subject.capitalize(), obj.capitalize())
            ]))

            KB.append(frozenset([
                atom(property_name, subject.capitalize())
            ]))
            continue

        m = re.fullmatch(r"(\w+) is (\w+)", s)

        if m:
            subject, predicate = m.groups()
            KB.append(frozenset([
                atom(predicate, subject.capitalize())
            ]))
            continue

        m = re.fullmatch(r"(\w+) (\w+) (\w+)", s)

        if m:
            subject, verb, obj = m.groups()
            KB.append(frozenset([
                atom(verb, subject.capitalize(), obj.capitalize())
            ]))
            continue

        raise ValueError("Unsupported sentence: " + sentence)

    return KB


def is_variable(x):
    return isinstance(x, str) and x.startswith("?")


def walk(x, theta):

    while is_variable(x) and x in theta:
        x = theta[x]

    return x


def unify(x, y, theta):

    x = walk(x, theta)
    y = walk(y, theta)

    if x == y:
        return theta

    if is_variable(x):
        theta[x] = y
        return theta

    if is_variable(y):
        theta[y] = x
        return theta

    return None


def unify_literals(l1, l2):

    sign1, pred1, args1 = l1
    sign2, pred2, args2 = l2

    if sign1 == sign2 or pred1 != pred2:
        return None

    if len(args1) != len(args2):
        return None

    theta = {}

    for a, b in zip(args1, args2):

        theta = unify(a, b, theta)

        if theta is None:
            return None

    return theta


def apply_substitution(clause, theta):

    result = set()

    for sign, pred, args in clause:

        new_args = tuple(walk(a, theta) for a in args)

        result.add((sign, pred, new_args))

    return frozenset(result)


counter = itertools.count()


def standardize(clause):

    number = next(counter)

    mapping = {}
    result = set()

    for sign, pred, args in clause:

        new_args = []

        for arg in args:

            if is_variable(arg):

                if arg not in mapping:
                    mapping[arg] = arg + "_" + str(number)

                new_args.append(mapping[arg])

            else:
                new_args.append(arg)

        result.add((sign, pred, tuple(new_args)))

    return frozenset(result)


def canonicalize(clause):

    mapping = {}
    result = set()

    for sign, pred, args in sorted(clause, key=repr):

        new_args = []

        for arg in args:

            if is_variable(arg):

                if arg not in mapping:
                    mapping[arg] = "?V" + str(len(mapping))

                new_args.append(mapping[arg])

            else:
                new_args.append(arg)

        result.add((sign, pred, tuple(new_args)))

    return frozenset(result)


def resolve(c1, c2):

    resolvents = set()

    c1 = standardize(c1)
    c2 = standardize(c2)

    for l1 in c1:

        for l2 in c2:

            theta = unify_literals(l1, l2)

            if theta is not None:

                remaining = (c1 - {l1}) | (c2 - {l2})

                new_clause = apply_substitution(
                    remaining, theta
                )

                new_clause = canonicalize(new_clause)

                resolvents.add(new_clause)

    return resolvents


def parse_query(query):

    m = re.fullmatch(
        r"\s*(\w+)\s+(\w+)\s+(\w+)\s*\.?\s*",
        query
    )

    if not m:
        raise ValueError(
            "Enter query in the form: John likes peanuts"
        )

    subject, verb, obj = m.groups()

    return atom(
        verb,
        subject.capitalize(),
        obj.capitalize()
    )


def resolution(KB, query, max_steps=1000):

    negated_query = (
        not query[0],
        query[1],
        query[2]
    )

    clauses = {
        canonicalize(c) for c in KB
    }

    clauses.add(
        canonicalize({negated_query})
    )

    print("\nStarting Resolution...")

    for step in range(max_steps):

        new = set()
        clause_list = list(clauses)

        for i in range(len(clause_list)):

            for j in range(i + 1, len(clause_list)):

                resolvents = resolve(
                    clause_list[i],
                    clause_list[j]
                )

                if frozenset() in resolvents:

                    print("\nEmpty clause obtained!")
                    print("Contradiction found.")
                    print("QUERY IS PROVED.")
                    return True

                new.update(resolvents)

        new -= clauses

        if not new:

            print("\nNo new clauses generated.")
            print("QUERY COULD NOT BE PROVED.")
            return False

        clauses.update(new)

        print("Resolution step:", step + 1)

    print("Maximum resolution steps reached.")
    return False


print("====================================")
print(" FIRST ORDER LOGIC - RESOLUTION")
print("====================================")

print("\nEnter Knowledge Base statements.")
print("Press ENTER on an empty line to finish.\n")

statements = []

while True:

    sentence = input("Enter statement: ")

    if sentence.strip() == "":
        break

    statements.append(sentence)

try:

    KB = create_kb(statements)

    print("\nKnowledge Base created successfully.")
    print("Total CNF clauses:", len(KB))

    query_text = input("\nEnter query: ")

    query = parse_query(query_text)

    result = resolution(KB, query)

    if result:
        print("\nFINAL RESULT: QUERY PROVED.")
    else:
        print("\nFINAL RESULT: QUERY NOT PROVED.")

except ValueError as error:

    print("\nInput Error:", error)